# Metody pro zlepšení lineární regrese

V tomto cvičení budeme porovnávat různé techniky, které umožňují dosáhnout lepšího výsledku lineární regrese na datasetu Boston Housing.

Připomeňme si, že model z kapitoly 3, trénovaný pouze na proměnných RM a LSTAT, dosáhl R² ≈ 0,65.

Postupně vyzkoušíme tyto přístupy:
* **Baseline** – model z kapitoly 3 (RM + LSTAT)
* **Všechny proměnné** – lineární model bez jakýchkoliv úprav
* **Korelace + VIF** – diagnostika multikolinearity, model bez kolineárních proměnných
* **RFE** – automatický výběr proměnných (Recursive Feature Elimination)
* **EFA** – redukce dimenzionality přes latentní faktory
* **PCA** – transformace do nekorelovaných komponent
* **Regularizace** – Ridge, Lasso, Elastic Net s křížovou validací pro výběr λ

Na konci porovnáme všechny modely v přehledné tabulce.

## Načtení a analýza dat

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

In [ ]:
data = pd.read_csv ("..\\dataset\\HousingData.csv")

In [ ]:
data=data.dropna()

In [ ]:
data.head()

In [ ]:
data.describe()

## Výchozí model: RM a LSTAT

Jako výchozí bod použijeme model z kapitoly 3, kde jsme na základě korelační analýzy vybrali proměnné **RM** (průměrný počet pokojů) a **LSTAT** (procento obyvatel s nižším socioekonomickým statusem).

Definujeme také pomocnou funkci `print_model_score` a seznam `results`, do kterého budeme průběžně ukládat výsledky všech modelů pro závěrečné porovnání.

Data rozdělíme na trénovací (80 %) a testovací (20 %) část. Nastavíme `random_state=42` pro reprodukovatelnost — stejné rozdělení bude použito ve všech modelech.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

results = []

def print_model_score(y_true, y_pred, label=""):
    """Print R² and RMSE; return (r2, rmse)."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    prefix = f"{label} " if label else ""
    print(f"{prefix}R²:   {r2:.4f}")
    print(f"{prefix}RMSE: {rmse:.4f}")
    return r2, rmse

X_base = data[['RM', 'LSTAT']].values
y_base = data['MEDV'].values
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

lr_base = LinearRegression()
lr_base.fit(X_train_base, y_train_base)

r2_tr, _ = print_model_score(y_train_base, lr_base.predict(X_train_base), "Train")
r2_te, rmse_te = print_model_score(y_test_base, lr_base.predict(X_test_base), "Test")
results.append({"Model": "Linear regression (RM+LSTAT)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## Lineární model ze všech proměnných

Pro srovnání vytvoříme model, který využívá všech 13 dostupných proměnných bez jakýchkoliv úprav.

In [ ]:
X = np.array(data.drop('MEDV', axis=1))
y = np.array(data['MEDV'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Vytvoření lineárního modelu

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

Vyhodnocení modelu na trénovacích datech

In [ ]:
y_pred = lr.predict(X_train)
r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

Vyhodnocení modelu na testovacích datech

In [ ]:
y_pred = lr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

In [ ]:
r2_all_tr = r2_score(y_train, lr.predict(X_train))
r2_all_te = r2_score(y_test, lr.predict(X_test))
rmse_all_te = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))
results.append({
    "Model": "Linear regression (all variables)",
    "Train R²": round(r2_all_tr, 4),
    "Test R²": round(r2_all_te, 4),
    "Test RMSE": round(rmse_all_te, 4),
})

## Korelace
Opět provedeme korelační analýzu a hledáme lineární závislosti mezi proměnnými.

In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111)
sns.heatmap(data.corr(),annot=True)

**Korelační matice** obsahuje Pearsonovy korelační koeficienty mezi všemi páry proměnných. Hodnoty se pohybují od -1 do 1.
* 1 → silná pozitivní lineární korelace
* -1 → silná negativní lineární korelace
* 0 → žádná lineární korelace

**Multikolinearita** nastává, když jsou predikční proměnné silně korelovány mezi sebou (např. DIS s INUDS, INOX, AGE).

Problém:
* Odhady regresních koeficientů jsou nestabilní a citlivé na malé změny dat.
* Interpretace jednotlivých koeficientů ztrácí smysl, protože nelze izolovat efekt jedné proměnné při fixních ostatních, pokud jsou navzájem korelované.
* Zvyšuje standardní chybu koeficientů, což snižuje statistickou významnost.


Důležitý je poslední řádek, který nám ukazuje lineární korelaci vysvětlujících proměnných a vysvětlované proměnné MEDV. Zdá se, že naše cílová proměnná je vysoce korelovaná s LSTAT a RM, což dává smysl, protože tyto dva faktory jsou velmi důležité pro tvorbu cen domů.  Je zde také mnoho multikolinearity.

Obvyklá interpretace regresního koeficientu je taková, že poskytuje odhad účinku jednotkové změny nezávislé proměnné na závislou proměnnou při zachování ostatních proměnných jako konstantních. V případě multikolinearity, to ale nemůžeme říci. Pokud je X1 v daném souboru dat silně korelována s jinou nezávislou proměnnou, X2, pak máme soubor pozorování, pro který X1 a X2 mají určitý lineární stochastický vztah. Nemůžeme tak zajistit při změně proměnné X1 X2 zůstane konstantní.

## Faktor inflace rozptylu (VIF) 

**Faktor inflace rozptylu (VIF)** detekuje multikolinearitu v regresní analýze. 

Její přítomnost může negativně ovlivnit výsledky regrese. VIF odhaduje, jak moc je rozptyl regresního koeficientu nadsazen v důsledku multikolinearity v modelu.

VIF=1/(1−R^2)

Kde R^2 je koeficient determinace. 

Zjednodušeně řečeno, je to podíl rozptylu nezávislé proměnné, který je vysvětlen závislou proměnnou. 

Provedeme tedy lineární regresi s použitím každé proměnné jako cíle a ostatních jako prediktorů a vypočítáme R^2 a poté pro ně vypočítáme VIF.

Pokud je VIF < 4, lze proměnnou použít, v opačném případě musíme najít způsob, jak odstranit kolinearitu těchto proměnných.

In [ ]:
vifdf = []
for i in data.columns:
    X = np.array(data.drop(i,axis=1))
    y = np.array(data[i])
    lr = LinearRegression()
    lr.fit(X,y)
    y_pred = lr.predict(X)
    r2 = r2_score(y,y_pred)
    vif = 1/(1-r2)
    vifdf.append((i,vif))

vifdf = pd.DataFrame(vifdf,columns=['Features','Variance Inflation Factor'])
vifdf.sort_values(by='Variance Inflation Factor')

Vidíme, že téměř polovina proměnných má  hodnotu VIF vyšší, nebo blízkou 4. TAX a RAD mají VIF téměř dvakrát vyšší, než je naše prahová hodnota.

Bude tedy vhodné vyřešit multikolinearitu. To lze dělat více způsoby:
* Odstranění korelovaných proměnných → vybrat jen jednu z páru silně korelovaných.
* Principal Component Analysis (PCA) → transformovat prediktory do ne-korelovaných komponent.
* Regularizace (Ridge, Lasso) → potlačuje vliv kolinearity a stabilizuje model.

### Model bez kolineárních proměnných

Odstraníme proměnné s příliš vysokým VIF (TAX, RAD) a natrénujeme nový lineární model. Tím ověříme, zda redukce multikolinearity sama o sobě zlepší predikci.

In [ ]:
features_novif = [c for c in data.columns if c not in ['MEDV', 'TAX', 'RAD']]
X_novif = data[features_novif].values
X_train_novif, X_test_novif, y_train_novif, y_test_novif = train_test_split(
    X_novif, data['MEDV'].values, test_size=0.2, random_state=42
)

lr_novif = LinearRegression()
lr_novif.fit(X_train_novif, y_train_novif)

r2_tr, _ = print_model_score(y_train_novif, lr_novif.predict(X_train_novif), "Train")
r2_te, rmse_te = print_model_score(y_test_novif, lr_novif.predict(X_test_novif), "Test")
results.append({"Model": "Linear regression (wihout TAX+RAD)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## Recursive Feature Elimination (RFE)

RFE je metoda automatického výběru proměnných. Funguje iterativně:
1. Natrénuje model na všech proměnných.
2. Odstraní proměnnou s nejmenším absolutním koeficientem.
3. Opakuje, dokud nezůstane požadovaný počet proměnných.

Na rozdíl od VIF (který řeší kolinearitu) RFE vybírá proměnné na základě jejich **přínosu pro predikci**.

In [ ]:
from sklearn.feature_selection import RFE

feature_names = [c for c in data.columns if c != 'MEDV']
X_rfe_raw = data[feature_names].values
X_train_rfe, X_test_rfe, y_train_rfe, y_test_rfe = train_test_split(
    X_rfe_raw, data['MEDV'].values, test_size=0.2, random_state=42
)

rfe = RFE(LinearRegression(), n_features_to_select=6)
rfe.fit(X_train_rfe, y_train_rfe)

selected_features = [feature_names[i] for i, s in enumerate(rfe.support_) if s]
print("Selected variables:", selected_features)

lr_rfe = LinearRegression()
lr_rfe.fit(X_train_rfe[:, rfe.support_], y_train_rfe)

r2_tr, _ = print_model_score(y_train_rfe, lr_rfe.predict(X_train_rfe[:, rfe.support_]), "Train")
r2_te, rmse_te = print_model_score(y_test_rfe, lr_rfe.predict(X_test_rfe[:, rfe.support_]), "Test")
results.append({"Model": "Linear regression + RFE (6 variables)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## Exploratory Factor Analysis (EFA)

EFA je metoda redukce dimenzionality, která hledá skryté (latentní) faktory vysvětlující korelační strukturu mezi proměnnými.

Srovnání s PCA:
* **PCA** maximalizuje vysvětlenou varianci dat — komponenty jsou lineárními kombinacemi proměnných.
* **EFA** modeluje korelační strukturu a předpokládá, že pozorované proměnné jsou způsobeny menším počtem skrytých faktorů.

Použijeme 3 faktory s rotací **varimax** (faktory zůstávají nekorelované), stejně jako v kapitole 2.

Standardizaci provedeme pouze na trénovacích datech (**fit na train, transform na test**), aby nedocházelo k úniku informací z testovací sady (data leakage).

In [ ]:
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

feature_names_efa = [c for c in data.columns if c != 'MEDV']
X_efa_raw = data[feature_names_efa].values
X_train_efa_raw, X_test_efa_raw, y_train_efa, y_test_efa = train_test_split(
    X_efa_raw, data['MEDV'].values, test_size=0.2, random_state=42
)

scaler_efa = StandardScaler()
X_train_efa_sc = scaler_efa.fit_transform(X_train_efa_raw)
X_test_efa_sc = scaler_efa.transform(X_test_efa_raw)

fa = FactorAnalysis(n_components=3, rotation='varimax', random_state=42)
X_train_fa = fa.fit_transform(X_train_efa_sc)
X_test_fa = fa.transform(X_test_efa_sc)

loadings = pd.DataFrame(
    fa.components_.T,
    index=feature_names_efa,
    columns=['Factor 1', 'Factor 2', 'Factor 3']
).round(3)
loadings

### Lineární model na základě EFA faktorů

Skóre faktorů (3 hodnoty místo 13 proměnných) použijeme jako vstup pro lineární regresi.

In [ ]:
lr_efa = LinearRegression()
lr_efa.fit(X_train_fa, y_train_efa)

r2_tr, _ = print_model_score(y_train_efa, lr_efa.predict(X_train_fa), "Train")
r2_te, rmse_te = print_model_score(y_test_efa, lr_efa.predict(X_test_fa), "Test")
results.append({"Model": "Linear regression + EFA (3 factors)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## PCA

### Myšlenka
* Pokud máme hodně vzájemně korelovaných proměnných, tak je v datech skrytá redundance.
* PCA tuto redundanci odstraní tím, že původní proměnné převede na nové, nekorelované proměnné = hlavní komponenty.
* Tyto komponenty jsou lineární kombinací původních proměnných.


### Jak PCA funguje? (intuice)
* Najde směr s největším rozptylem dat (1. hlavní komponenta).
* Najde druhý směr s co největším rozptylem, ale ortogonální k prvnímu (2. hlavní komponenta).
* Pokračuje, dokud nevyčerpá všechny dimenze.
* Výsledek:
    * Hlavní komponenty jsou nekorelované.
    * První komponenty vysvětlují většinu variability v datech.


### K čemu PCA slouží
* Odstranění multikolinearity → komponenty jsou ortogonální → žádná kolinearita.
* Redukce dimenzionality → necháme si jen prvních pár komponent, které vysvětlí např. 90–95 % variability.
* Vizualizace → složitá data z mnoha proměnných lze vykreslit do 2D/3D prostoru.


PCA je citlivá na měřítko proměnných. Proto se před aplikací PCA obvykle dělá standardizace.

PCA nebudeme psát ručně, ale použijeme její implementaci z knihovny.

In [ ]:
from sklearn.decomposition import PCA

Počet PCA komponent bude 13 stejně jako vstupních parametrů.

Výstupní MEDV musíme ze vstupu do PCA odstranit.

Data musí být standardizovaná.

In [ ]:
feature_names_pca = [c for c in data.columns if c != 'MEDV']
X_pca_raw = data[feature_names_pca].values
scaler_pca = StandardScaler()
X_pca_sc = scaler_pca.fit_transform(X_pca_raw)

In [ ]:
pca = PCA(n_components=13)
X_pca = pca.fit_transform(X_pca_sc)

In [ ]:
X_pca_all = pd.DataFrame(X_pca,columns=['PCA1','PCA2','PCA3','PCA4','PCA5','PCA6','PCA7','PCA8','PCA9','PCA10','PCA11','PCA12','PCA13'])

Distribuční funkce PCA proměnných jsou odlišné od těch původních.

In [ ]:
pos = 1
fig = plt.figure(figsize=(12,16))
for i in X_pca_all.columns:
    ax = fig.add_subplot(7,2, pos)
    pos = pos + 1
    sns.histplot(X_pca_all[i],ax=ax, kde=True)

PCA měla redukovat multikolinearitu. Tak si to zkontrolujeme.

V korelační matici je vidět, že PCA komponenty nejsou na sobě závislé — to je klíčová vlastnost PCA.

**Proč MEDV nekoreluje nejvíce s PCA1?**
PCA seřazuje komponenty podle toho, kolik vysvětlují variance **vstupních proměnných X** — ne podle korelace s cílovou proměnnou MEDV. Proto první komponenta vysvětluje nejvíce variability v datech, ale nemusí být nejlépe predikovat MEDV.

Model trénovaný na prvních 6 komponentách (dle pořadí z PCA, tj. nejvyšší vysvětlená variance) tak nemusí být optimální. Pokud bychom chtěli optimalizovat výběr pro MEDV, použili bychom jinou metodu (např. PLS — Partial Least Squares).

In [ ]:
data_pca_all = pd.DataFrame(
    X_pca,
    columns=[f'PCA{i}' for i in range(1, 14)]
)
data_pca_all['MEDV'] = y

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111)
sns.heatmap(data_pca_all.corr(), annot=True)

## Lineární model ze všech PCA proměnných 
Nyní si vytvoříme nový data s hlavními komponentami jako vstupní proměnné a MEDV jako výstupní proměnnou.

Data si opět rozdělíme na na trénovací a testovací.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca_all.values, y, test_size=0.2, random_state=42
)

Vytvoříme lineární model

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

Validace modelu pro trénovací data

In [ ]:
y_pred = lr.predict(X_train)
r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

Validace modelu pro testovací data

In [ ]:
y_pred = lr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

In [ ]:
r2_pca_tr = r2_score(y_train, lr.predict(X_train))
r2_pca_te = r2_score(y_test, lr.predict(X_test))
rmse_pca_te = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))
results.append({
    "Model": "Linear regression + PCA (13 komponent)",
    "Train R²": round(r2_pca_tr, 4),
    "Test R²": round(r2_pca_te, 4),
    "Test RMSE": round(rmse_pca_te, 4),
})

Výsledný model z PCA proměnných je o něco lepší než původní lineární model z původních proměnných.

## Lineární model ze 6 PCA proměnných
PCA lze také použít pro redukci dimenzionality. 

Vytvoříme tedy model, který bude mít místo 13 vstupních proměnných pouze 6 proměnných.

In [ ]:
lr = LinearRegression()
lr.fit(X_train[:, :6], y_train)

Validace modelu na trénovacích datech

In [ ]:
y_pred = lr.predict(X_train[:,0:6])
r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

Validace modelu na testovacích datech

In [ ]:
y_pred = lr.predict(X_test[:,0:6])
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

In [ ]:
r2_pca6_tr = r2_score(y_train, lr.predict(X_train[:, :6]))
r2_pca6_te = r2_score(y_test, lr.predict(X_test[:, :6]))
rmse_pca6_te = np.sqrt(mean_squared_error(y_test, lr.predict(X_test[:, :6])))
results.append({
    "Model": "Linear regression + PCA (6 components)",
    "Train R²": round(r2_pca6_tr, 4),
    "Test R²": round(r2_pca6_te, 4),
    "Test RMSE": round(rmse_pca6_te, 4),
})

Přesnost redukovaného modelu je o podle očekávání o něco nižší. Na druhou stranu model je menší.

## Regularizace

Regularizace přidává penalizační člen k ztrátové funkci lineární regrese. Tím potlačuje příliš velké koeficienty a snižuje riziko přeučení.

| Metoda | Penalizace | Efekt |
|--------|-----------|-------|
| **Ridge (L2)** | součet čtverců koeficientů | Zachovává všechny proměnné, zmenšuje jejich vliv |
| **Lasso (L1)** | součet absolutních hodnot koeficientů | Vynuluje nepodstatné koeficienty → automatický výběr proměnných |
| **Elastic Net** | kombinace L1 + L2 | `l1_ratio` určuje poměr; vhodné při skupinách korelovaných proměnných |

Výši penalizace určuje parametr **λ** (v scikit-learn `alpha`). Optimální hodnotu najdeme křížovou validací.

**Důležité:** Regularizace je citlivá na měřítko — data musíme nejdříve standardizovat. Scaler fitujeme pouze na trénovací sadě.

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_names_reg = [c for c in data.columns if c != 'MEDV']
X_reg = data[feature_names_reg].values
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, data['MEDV'].values, test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_train_sc = scaler_reg.fit_transform(X_train_reg)
X_test_sc = scaler_reg.transform(X_test_reg)

### Ridge (L2 regularizace)

`RidgeCV` prohledává zadané hodnoty `alpha` a vybere tu nejlepší pomocí křížové validace.

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge.fit(X_train_sc, y_train_reg)

print(f"Optimal alpha: {ridge.alpha_:.4f}")
r2_tr, _ = print_model_score(y_train_reg, ridge.predict(X_train_sc), "Train")
r2_te, rmse_te = print_model_score(y_test_reg, ridge.predict(X_test_sc), "Test")
results.append({"Model": "Ridge CV", "Train R²": r2_tr, "Test R²": r2_te, "Test RMSE": rmse_te})

### Lasso (L1 regularizace)

Lasso může vynulovat koeficienty nepodstatných proměnných — implicitně provádí výběr proměnných. `LassoCV` hledá optimální `alpha` křížovou validací.

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(alphas=np.logspace(-3, 1, 100), cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train_sc, y_train_reg)

print(f"Optimal alpha: {lasso.alpha_:.4f}")
print(f"Non-zero coeficients: {np.sum(lasso.coef_ != 0)} / {X_train_sc.shape[1]}")
r2_tr, _ = print_model_score(y_train_reg, lasso.predict(X_train_sc), "Train")
r2_te, rmse_te = print_model_score(y_test_reg, lasso.predict(X_test_sc), "Test")
results.append({"Model": "Lasso CV", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

### Elastic Net

Elastic Net kombinuje L1 a L2 penalizaci. `ElasticNetCV` prohledává kombinace `alpha` a `l1_ratio` (poměr L1 ku L2 — 0 = Ridge, 1 = Lasso) pomocí křížové validace.

In [ ]:
from sklearn.linear_model import ElasticNetCV

elastic = ElasticNetCV(
    alphas=np.logspace(-3, 1, 50),
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
    cv=5,
    random_state=42,
    max_iter=10000,
)
elastic.fit(X_train_sc, y_train_reg)

print(f"Optimal alpha:    {elastic.alpha_:.4f}")
print(f"Optimal l1_ratio: {elastic.l1_ratio_:.2f}")
r2_tr, _ = print_model_score(y_train_reg, elastic.predict(X_train_sc), "Train")
r2_te, rmse_te = print_model_score(y_test_reg, elastic.predict(X_test_sc), "Test")
results.append({"Model": "Elastic Net CV", "Train R²": r2_tr, "Test R²": r2_te, "Test RMSE": rmse_te})

## Srovnání modelů

Výsledky všech modelů seřazené podle testovacího R². Vyšší R² a nižší RMSE znamená lepší predikci na nových datech.

Modely, které používají více proměnných jsou na testovacích datech přesnější než zjednodušené modely používající jen některá data.

In [ ]:
pd.DataFrame(results).round(4).sort_values("Test R²", ascending=False).reset_index(drop=True)